# Display Degraded LSST Galaxy Images
This notebook reads the degraded 5-band FITS cubes (`(5, 41, 41)` shape) from the `../data/final-sample-output` directory and displays some examples in each band along with an RGB composite.

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import make_lupton_rgb

### 1. Locate Degraded FITS Files

In [ ]:
degraded_dir = "../data/final-sample-output/simct_images"
fits_files = sorted(glob.glob(os.path.join(degraded_dir, "**/*.fits"), recursive=True))
print(f"Found {len(fits_files)} degraded FITS files in {degraded_dir}.")
print("\nFirst few files:")
for f in fits_files[:5]:
    print(f)

### 2. Define Example Display Function

In [ ]:
def display_degraded_examples(fits_paths, num_examples=6):
    if not fits_paths:
        print("No FITS files found to display.")
        return
        
    num_to_show = min(len(fits_paths), num_examples)
    for idx in range(num_to_show):
        path = fits_paths[idx]
        sample_name = os.path.basename(os.path.dirname(path))
        filename = os.path.basename(path)
        
        print(f"\n{'='*80}\nExample {idx+1}: Sample={sample_name} | File={filename}\n{'='*80}")
        
        try:
            with fits.open(path) as hdul:
                cube = hdul[0].data.astype(np.float32)
                
            print(f"Cube Shape: {cube.shape} (5 bands, height, width)")
            
            fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
            bands = ['g', 'r', 'i', 'z', 'y']
            
            def _norm(arr):
                p1, p99 = np.nanpercentile(arr, [1, 99])
                # Prevent divide-by-zero
                denom = p99 - p1 if p99 > p1 else 1.0
                return np.clip((arr - p1) / denom, 0, 1)
                
            for j, b in enumerate(bands):
                axes[j].imshow(_norm(cube[j]), cmap="gray", origin="lower")
                axes[j].set_title(b)
                axes[j].axis("off")
                
            # Create RGB composite using i, r, g bands (indices 2, 1, 0)
            rgb = make_lupton_rgb(cube[2], cube[1], cube[0], Q=10, stretch=0.5)
            axes[5].imshow(rgb, origin="lower")
            axes[5].set_title("i/r/g RGB")
            axes[5].axis("off")
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Error displaying {filename}: {e}")

### 3. Display Examples

In [ ]:
display_degraded_examples(fits_files, num_examples=6)

# 3. Visualize Failed Degradation Samples
This section parses `../data/final-sample-output/degrade.log` to find galaxies that failed the degradation pipeline. It loads their original images and runs the degradation in a "relaxed/debug" mode (e.g. bypassing magnitude thresholds or disabling background noise addition if the input is already noisier than LSST target) so you can inspect the original and degraded versions side-by-side.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath("HSC_to_LSST"))
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from hsc_to_lsst.pipeline import zero_point_change
from hsc_to_lsst.lsst_props import dp0_gmm_sampler
from hsc_to_lsst.pipeline import HSC_MEAN_FWHM, LSST_SINGLE_EXP_AVG_ZERO_POINTS
from hsc_to_lsst.data_degradation.hsc_degradation import hsc_to_lsst

# 1. Parse degrade.log
log_path = "../data/final-sample-output/degrade.log"
failed_galaxies = {}
if os.path.exists(log_path):
    with open(log_path) as f:
        for line in f:
            if ":" in line:
                parts = line.split(":")
                gid = parts[0].strip()
                err = parts[1].strip()
                if gid.isdigit():
                    if gid not in failed_galaxies:
                        failed_galaxies[gid] = []
                    failed_galaxies[gid].append(err)
    print(f"Loaded {len(failed_galaxies)} failed galaxies from log.")
else:
    print(f"Warning: {log_path} not found.")

# 2. Match failed galaxies with catalog parameters to find their sample and coordinates
samples = ['hectomap', 'udeep', 'w03_gama09h', 'w05_vvds', 'w07_aegis', 'xmm']
data_dir = "../data/final-sample"
failed_list = []

for s in samples:
    csv_path = os.path.join(data_dir, "params", f"{s}_finalpar.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path, dtype={"galaxy_id": str})
        matches = df[df["galaxy_id"].isin(failed_galaxies.keys())]
        for idx, row in matches.iterrows():
            failed_list.append({
                "galaxy_id": row["galaxy_id"],
                "sample_name": s,
                "ra": row["galaxy_ra"],
                "dec": row["galaxy_dec"],
                "errors": failed_galaxies[row["galaxy_id"]]
            })

print(f"Matched {len(failed_list)} failed galaxies with parameters.")

# Coordinate converter helper
def _coord_str(ra, dec):
    ra_h = ra / 15.0
    h = int(ra_h); m = int((ra_h - h) * 60); s = (ra_h - h - m / 60) * 3600
    sign = "+" if dec >= 0 else "-"
    adec = abs(dec)
    dh = int(adec); dm = int((adec - dh) * 60); ds = (adec - dh - dm / 60) * 3600
    return f"{h:02d}{m:02d}{s:06.3f}{sign}{dh:02d}{dm:02d}{ds:06.3f}"

# 3. Define debug degradation and display function
def display_failed_galaxy_debug(failed_item, coadd_years=1):
    gid = failed_item["galaxy_id"]
    sname = failed_item["sample_name"]
    ra = failed_item["ra"]
    dec = failed_item["dec"]
    errors = failed_item["errors"]
    
    coord = _coord_str(ra, dec)
    sim_path = os.path.join(data_dir, "simct_images", sname, f"imoutf_{coord}.fits")
    
    if not os.path.exists(sim_path):
        print(f"Original image file not found for galaxy {gid} at: {sim_path}")
        return
        
    print(f"\n{'='*100}")
    print(f"DEBUGGING GALAXY {gid} (Sample: {sname})")
    print(f"RA={ra:.5f}, Dec={dec:.5f}")
    print("Logged Pipeline Errors:")
    for err in errors:
        print(f"  - {err}")
    print(f"{'='*100}")
    
    try:
        # Load original image
        with fits.open(sim_path) as hdul:
            cube = hdul[0].data.astype(np.float32)
            
        bands = ['g', 'r', 'i', 'z', 'y']
        
        # Initialize sampler
        dp0_sampler = dp0_gmm_sampler(str(coadd_years))
        dp0_stats = [dp0_sampler[b].sample() for b in bands]
        dp0_rms = [dp0_stats[b]['rms'] for b in range(5)]
        dp0_zero_points = [dp0_stats[b]['zero_point'] for b in range(5)]
        
        # Perform magnitude scaling
        images_list = [cube[b] for b in range(5)]
        scaled_images, mag_change = zero_point_change(
            images_list, 27, dp0_zero_points, dp0_rms, rms_frac_thresh=0.3
        )
        
        print(f"Magnitude change required: {mag_change:.3f} mag (Threshold: 2.0)")
        zp_status = "Pass" if mag_change <= 2.0 else "Fail (exceeds 2.0 limit)"
        print(f"Zero point scaling status: {zp_status}")
        
        # Run band degradation with relaxed safety checks
        degraded_images = []
        band_statuses = []
        
        for b, band in enumerate(bands):
            status = "Success"
            try:
                deg_img = hsc_to_lsst(
                    scaled_images[b],
                    band,
                    exp_time=dp0_stats[b]['exp_time'],
                    lsst_zero_point=dp0_stats[b]['zero_point'],
                    lsst_single_exposure_zero_point=LSST_SINGLE_EXP_AVG_ZERO_POINTS[band],
                    lsst_ccd_gain=0.7,
                    hsc_fwhm=HSC_MEAN_FWHM[band],
                    hsc_psf=None,
                    hsc_pix_scale=0.168,
                    lsst_fwhm=dp0_stats[b]['seeing'],
                    background_noise=dp0_stats[b]['rms'],
                    background_median=dp0_stats[b]['median'],
                    psf_transform=True,
                    add_poisson_noise=True,
                    add_background_noise=True,
                    use_noise_diff=True,
                    out_size=41
                )
            except ValueError as e:
                if "Background noise is larger than target" in str(e):
                    status = "Fail: Noise (Original > LSST)"
                    # Fallback to no background noise addition to show resampled structure
                    try:
                        deg_img = hsc_to_lsst(
                            scaled_images[b],
                            band,
                            exp_time=dp0_stats[b]['exp_time'],
                            lsst_zero_point=dp0_stats[b]['zero_point'],
                            lsst_single_exposure_zero_point=LSST_SINGLE_EXP_AVG_ZERO_POINTS[band],
                            lsst_ccd_gain=0.7,
                            hsc_fwhm=HSC_MEAN_FWHM[band],
                            hsc_psf=None,
                            hsc_pix_scale=0.168,
                            lsst_fwhm=dp0_stats[b]['seeing'],
                            background_noise=dp0_stats[b]['rms'],
                            background_median=dp0_stats[b]['median'],
                            psf_transform=True,
                            add_poisson_noise=True,
                            add_background_noise=False,
                            out_size=41
                        )
                    except Exception as e2:
                        status = f"Error: {e2}"
                        deg_img = np.zeros((41, 41))
                else:
                    status = f"Error: {e}"
                    deg_img = np.zeros((41, 41))
            except Exception as e:
                status = f"Error: {e}"
                deg_img = np.zeros((41, 41))
                
            degraded_images.append(deg_img)
            band_statuses.append(status)
            
        # Plotting original vs degraded
        fig, axes = plt.subplots(2, 5, figsize=(20, 8.5))
        
        def _norm(arr):
            p1, p99 = np.nanpercentile(arr, [1, 99])
            denom = p99 - p1 if p99 > p1 else 1.0
            return np.clip((arr - p1) / denom, 0, 1)
            
        for i, band in enumerate(bands):
            # Original band
            orig_img = cube[i]
            axes[0, i].imshow(_norm(orig_img), origin='lower', cmap='gray')
            axes[0, i].set_title(f"Original {band}")
            axes[0, i].axis("off")
            
            # Degraded band (relaxed)
            deg_img = degraded_images[i]
            axes[1, i].imshow(_norm(deg_img), origin='lower', cmap='gray')
            axes[1, i].set_title(f"LSST {band}\n({band_statuses[i]})", color='green' if band_statuses[i] == 'Success' else 'red')
            axes[1, i].axis("off")
            
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error debugging galaxy {gid}: {e}")

In [ ]:
# Display first few failed examples
if failed_list:
    for item in failed_list[:3]:
        display_failed_galaxy_debug(item)
else:
    print("No failed galaxies found to display.")